# SUBCPMK-3 — Ekstraksi 50 Putusan dan Crawler Metadata Direktori MA PN Kuala Kurun

**Kelompok**: Rizqullah Zuhayr Alfito Khan (202310370311124) dan Fakhri Ziddan Akbar (202310370311143)  
**Sumber data**: ZIP 50 PDF yang sudah dikumpulkan pada SUBCPMK-2.  
**Tujuan**: ekstraksi PDF → TXT → pemisahan **amar putusan** dan **barang bukti** berdasarkan struktur dokumen, validasi otomatis yang lebih ketat, dan ekspor CSV (5 kolom sesuai dosen).

> **Cakupan:** Notebook ini hanya memproses berkas milik kelompok. **Belum mencakup web crawler langsung** ke situs Direktori Putusan MA, sehingga komponen crawling SUBCPMK-3 tetap belum terpenuhi. Status BHT dan ketepatan kutipan tetap perlu verifikasi terhadap sumber resmi.

## Perbaikan termasuk tahap pembersihan akhir

- Menargetkan heading `MENGADILI` sebagai **baris tersendiri**, bukan frasa `mengadili perkara` pada paragraf pembuka.
- Mengambil isi amar **sampai sebelum** `Demikianlah diputuskan` (bukan batas 25.000 karakter).
- Mengambil barang bukti **dari amar**, tidak dari daftar barang bukti di pendahuluan.
- Membersihkan disclaimer situs MA dan header/footer berulang **hanya setelah** bagian relevan diisolasi.
- Menambahkan audit anomali (disclaimer tersisa, kolom kosong, amar tidak berurutan, kolom terlalu panjang) yang **tidak menyamakan nonkosong dengan benar**.

## 1. Instalasi pustaka

In [ ]:
!pip -q install pymupdf pandas

## 2. Memuat ZIP

Pilih satu ZIP: `Narkotika.zip` (berisi PDF) atau `hasil_ekstraksi_TXT.zip` (TXT keluaran Colab sebelumnya). ZIP **PDF lebih disarankan**, karena langkah konversi PDF→TXT bisa direproduksi. Jika memakai Google Drive, ganti `USE_DRIVE = True` dan ubah path.

In [ ]:
from pathlib import Path
import os, re, zipfile, shutil, json
import pandas as pd
import pymupdf

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/Narkotika.zip'  # sesuaikan
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Pilih ZIP PDF atau TXT terlebih dahulu.')
    ZIP_PATH = '/content/' + next(iter(uploaded.keys()))

ROOT = Path('/content/SUBCPMK3_REVISI')
PDF_DIR, TXT_DIR, OUT_DIR = ROOT/'PDF', ROOT/'TXT', ROOT/'Output'
for d in (PDF_DIR, TXT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)
if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError('Arsip bukan ZIP valid; kompres ulang PDF asli.')
with zipfile.ZipFile(ZIP_PATH) as archive:
    bad = archive.testzip()
    if bad:
        raise RuntimeError(f'Arsip tidak utuh: {bad}')
    input_entries = [n for n in archive.namelist()
                     if n.lower().endswith(('.pdf', '.txt')) and not n.endswith('/')]
    if not input_entries:
        raise RuntimeError('ZIP tidak memuat PDF atau TXT.')
    out_names = [Path(n).name for n in input_entries]
    if len(out_names) != len(set(out_names)):
        raise RuntimeError('Ada nama file duplikat di subfolder ZIP; mohon perbaiki.')
    for member in input_entries:
        filename = Path(member).name
        target = (PDF_DIR if filename.lower().endswith('.pdf') else TXT_DIR) / filename
        with archive.open(member) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

pdf_files = sorted(PDF_DIR.glob('*.pdf'))
txt_files = sorted(TXT_DIR.glob('*.txt'))
print('PDF:', len(pdf_files), '| TXT awal:', len(txt_files))

## 3. Konversi PDF menjadi TXT (jika input PDF)

In [ ]:
conversion_log=[]
for pdf in pdf_files:
    try:
        doc = pymupdf.open(pdf)
        text = '\n\n'.join(page.get_text('text') for page in doc)
        target = TXT_DIR / f'{pdf.stem}.txt'
        target.write_text(text, encoding='utf-8')
        conversion_log.append({'file':pdf.name,'halaman':len(doc),
                               'karakter':len(text),'status':'OK' if text.strip() else 'KOSONG / BUTUH OCR'})
        doc.close()
    except Exception as e:
        conversion_log.append({'file':pdf.name,'halaman':None,'karakter':0,
                               'status':f'GAGAL: {e}'})
if conversion_log:
    display(pd.DataFrame(conversion_log).head())
txt_files = sorted(TXT_DIR.glob('*.txt'))
print('TXT tersedia:',len(txt_files))
if not txt_files:
    raise RuntimeError('Tidak ada TXT yang berhasil disiapkan.')

## 4. Fungsi ekstraksi dan sanitasi watermark/disclaimer

In [ ]:
# Gunakan satu baris MENGADILI agar tidak salah mengambil frase "mengadili perkara".
PAT_AMAR = re.compile(r'(?im)^\s*M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*$')
PAT_AKHIR = re.compile(r'(?i)\bDemikian(?:lah)?\s+diputus(?:kan)?\b')
PAT_BUKTI = re.compile(r'(?i)(?:menetapkan|menyatakan)\s+barang\s+bukti\s+berupa\s*[:;]?')
PAT_BIAYA = re.compile(r'(?im)^\s*(?:\d+[.)]\s*)?(?:Membebankan|Menetapkan biaya perkara|Menghukum Terdakwa membayar biaya perkara)\b')
PAT_NOMOR = re.compile(r'(?i)(\d+\s*/\s*Pid\.?\s*Sus\.?\s*/\s*20\d{2}\s*/\s*PN\s*Kkn)')
PAT_DISCLAIMER = re.compile(
    r'(?is)Disclaimer\s+Kepaniteraan Mahkamah Agung Republik Indonesia.*?\(ext\.\s*318\)')
PAT_FOOTER = re.compile(r'(?im)^\s*Halaman\s+\d+\s+(?:dari\s+\d+\s+Putusan\s+Nomor[^\n]*)?\s*$')
PAT_HEADER = re.compile(r'(?im)^\s*(?:Direktori Putusan Mahkamah Agung Republik Indonesia|putusan\.mahkamahagung\.go\.id)\s*$')

def clean_section(text):
    text = PAT_DISCLAIMER.sub(' ', text)
    text = PAT_FOOTER.sub(' ', text)
    text = PAT_HEADER.sub(' ', text)
    # Bersihkan fragmen watermark/footer yang tersisa akibat pemotongan halaman.
    # Hanya hapus baris yang cocok persis dengan pola artefak, bukan kalimat hukum.
    text = re.sub(r'(?im)^\s*mah Agung Republik Indonesia\s*$', '', text)
    text = re.sub(
        r'(?im)^\s*Hal\.\s*\d+\s+dari\s+\d+\s+hal\.\s*Putusan\s+Nomor\s+\d+\s*/\s*Pid\.Sus\s*/\s*\d{4}\s*/\s*PN\s*Kkn\s*$',
        '', text
    )
    # Bersihkan baris watermark singkat yang berdiri sendiri, bukan isi hukum.
    text = re.sub(r'(?im)^\s*(?:hkama|ahkamah Agung Repub|ahkamah Agung Republik Indonesia|blik Indonesi)\s*$', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def extract_parts(raw):
    matches = list(PAT_AMAR.finditer(raw))
    if not matches:
        return '', '', 'Tidak ditemukan heading MENGADILI yang berdiri sendiri'
    start = matches[-1].end()
    after = raw[start:]
    end = PAT_AKHIR.search(after)
    if not end:
        return '', '', 'Tidak ditemukan batas akhir amar (Demikianlah diputuskan)'
    amar_raw = after[:end.start()]
    candidates = list(PAT_BUKTI.finditer(amar_raw))
    if not candidates:
        return clean_section(amar_raw), '', 'Tidak ditemukan Barang Bukti pada bagian amar'
    evidence_text = amar_raw[candidates[-1].end():]
    stop = PAT_BIAYA.search(evidence_text)
    if stop:
        evidence_text = evidence_text[:stop.start()]
    else:
        # Terdapat variasi penomoran; jika tak ditemukan, tetap simpan tapi beri flag.
        return clean_section(amar_raw), clean_section(evidence_text), 'Batas akhir barang bukti belum terdeteksi'
    return clean_section(amar_raw), clean_section(evidence_text), ''

def get_case_no(raw, filename):
    first = raw[:12000]
    match = PAT_NOMOR.search(first)
    if not match:
        match = PAT_NOMOR.search(raw)
    if match:
        # Normalisasi spasi tanpa mengubah angka/kode.
        return re.sub(r'\s+', '', match.group(1)).replace('PNKkn','PN Kkn')
    stem = Path(filename).stem
    m = re.search(r'putusan_(\d+)_pid\.sus_(20\d{2})_pn_kkn', stem, re.I)
    return f'{m.group(1)}/Pid.Sus/{m.group(2)}/PN Kkn' if m else ''

print('Fungsi revisi siap.')

## 5. Ekstraksi seluruh dokumen dari TXT

In [ ]:
records=[]
for i, txt in enumerate(txt_files,1):
    raw = txt.read_text(encoding='utf-8', errors='replace')
    amar, barang_bukti, warning = extract_parts(raw)
    nomor = get_case_no(raw,txt.name)
    # Nama PN ditetapkan berdasarkan koleksi yang dipilih, dan tetap diperiksa silang dari teks.
    label_pn = 'PN Kuala Kurun' if re.search(r'Pengadilan Negeri Kuala Kurun', raw, re.I) else ''
    flags=[]
    if warning: flags.append(warning)
    if not nomor: flags.append('Nomor putusan tidak ditemukan')
    if not label_pn: flags.append('PN Kuala Kurun tidak ditemukan dalam dokumen')
    if not amar: flags.append('Amar kosong')
    if not barang_bukti: flags.append('Barang bukti kosong')
    if len(amar)>15000: flags.append('Amar terlalu panjang')
    if len(barang_bukti)>10000: flags.append('Barang bukti terlalu panjang')
    for title,value in [('Amar',amar),('Barang Bukti',barang_bukti)]:
        if re.search(r'Disclaimer|Kepaniteraan Mahkamah Agung RI melalui|Halaman \d+ dari \d+',value,re.I):
            flags.append(f'{title} masih memiliki noise header/disclaimer')
    records.append({'No':i,'No Putusan':nomor,'Lembaga Peradilan':label_pn,
                    'Barang Bukti':barang_bukti,'Amar Putusan':amar,
                    'Nama File':txt.name,'Catatan QA':'; '.join(flags),
                    'Pemeriksaan Manual': 'PERLU CEK' if flags else 'CEK SAMPEL / SUMBER'})

full_df=pd.DataFrame(records)
main_cols=['No','No Putusan','Lembaga Peradilan','Barang Bukti','Amar Putusan']
display(full_df[main_cols].head(3))
print('Dokumen:',len(full_df),'| Bendera anomali:',int(full_df['Catatan QA'].ne('').sum()))

## 6. Pemeriksaan mutu yang tidak hanya mengecek kolom kosong

**Penting:** `Catatan QA` kosong tidak berarti ekstraksi pasti benar. Cocokkan sejumlah dokumen yang mewakili format berbeda dengan PDF asli. Periksa bahwa seluruh amar termasuk putusan inti dan bahwa daftar barang bukti tidak terpotong atau melebar ke biaya perkara.

In [ ]:
required=['No Putusan','Lembaga Peradilan','Barang Bukti','Amar Putusan']
summary=pd.DataFrame([{'kolom':c,
                       'terisi':int(full_df[c].astype(str).str.strip().ne('').sum()),
                       'kosong':int(full_df[c].astype(str).str.strip().eq('').sum())} for c in required])
display(summary)
print('Catatan QA ditemukan pada',int(full_df['Catatan QA'].ne('').sum()),'dokumen')
display(full_df.loc[full_df['Catatan QA'].ne(''), ['No','No Putusan','Nama File','Catatan QA']])

assert len(full_df)==len(txt_files), 'Ada file TXT yang tidak masuk ke tabel.'
if full_df['No Putusan'].duplicated().any():
    print('PERINGATAN: nomor perkara berulang!')

### 6A. Audit residu watermark dan footer
Pemeriksaan ini mendeteksi teks sampah setelah ekstraksi. Ia **tidak** menyatakan akurasi 100% terhadap putusan asli.

In [ ]:
# Pemeriksaan residu setelah pembersihan (bukan validasi kebenaran hukum)
noise_pattern = re.compile(
    r'(?i)mah Agung Republik Indonesia|Hal\.\s*\d+\s+dari\s+\d+\s+hal\.|Disclaimer'
)
for col in ['Barang Bukti','Amar Putusan']:
    residue = full_df[col].fillna('').astype(str).str.contains(noise_pattern).sum()
    print(f'Residu watermark/footer pada {col}: {int(residue)}')
    if residue:
        display(full_df.loc[full_df[col].fillna('').astype(str).str.contains(noise_pattern),
                            ['No','No Putusan','Nama File','Catatan QA']])
print('Nomor putusan unik:', full_df['No Putusan'].nunique(), '/', len(full_df))
print('Kolom kosong:', {c: int(full_df[c].astype(str).str.strip().eq('').sum()) for c in required})
print('CATATAN: Tidak ada residu bukan berarti isi amar atau barang bukti telah benar secara hukum. Periksa PDF asli.')

## 7. Simpan CSV, audit, dan ZIP TXT untuk pengumpulan

In [ ]:
main_csv=OUT_DIR/'hasil_ekstraksi_putusan_BERSIH.csv'
audit_csv=OUT_DIR/'laporan_validasi_BERSIH.csv'
full_df[main_cols].to_csv(main_csv,index=False,encoding='utf-8-sig')
full_df.to_csv(audit_csv,index=False,encoding='utf-8-sig')
txt_zip=shutil.make_archive(str(OUT_DIR/'TXT_50_Putusan'),'zip',TXT_DIR)
print('Selesai:',main_csv, audit_csv, txt_zip, sep='\n')

## 8. Unduh hasil

In [ ]:
from google.colab import files
files.download(str(main_csv))
files.download(str(audit_csv))
files.download(txt_zip)

## 9. Ringkasan hasil ekstraksi lokal

Tahap PDF/TXT ke CSV selesai di atas. Bagian tambahan di bawah menelusuri metadata publik di situs MA hanya jika akses otomatis diizinkan. Data situs dan data hasil ekstraksi PDF **tidak otomatis dianggap sama**; pencocokan menggunakan nomor putusan.

In [ ]:
print(json.dumps({'jumlah_txt':len(txt_files),'jumlah_csv':len(full_df),
                  'jumlah_anomali_otomatis':int(full_df['Catatan QA'].ne('').sum()),
                  'pemeriksaan_substansi_pdf':'BELUM (harus dilakukan manual)',
                  'crawling_website':'BELUM'},ensure_ascii=False,indent=2))

---
## 10. Web crawler metadata Direktori Putusan Mahkamah Agung

**Alur:** pemeriksaan robots.txt → akses halaman direktori resmi → temukan tautan klasifikasi dan putusan → ekstraksi metadata publik → filter PN Kuala Kurun → cocokkan dengan 50 nomor putusan dari CSV → simpan CSV hasil crawling serta laporan aktivitas.

**Batasan penting:** Situs bisa mengembalikan `403`, `429`, atau menolak crawler. Notebook ini **tidak melewati** pembatasan, CAPTCHA, atau login. Bila `robots.txt` tidak dapat dipastikan, akses otomatis berhenti (fail-closed), lalu laporan kendala tetap dibuat. Hasil crawling dapat berjumlah 0 meski CSV lokal lengkap 50 baris. **Tidak** mengunduh berkas putusan atau lampiran melalui crawler.

URL awal yang digunakan merupakan halaman direktori MA dan daftar klasifikasi yang terverifikasi; tautan menuju PN Kuala Kurun dicari dari HTML, bukan mengarang URL.

In [ ]:
!pip -q install requests beautifulsoup4 lxml

### 10A. Konfigurasi dan pengecekan izin robot

In [ ]:
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
from datetime import datetime, timezone
import requests, time, re, json
from pathlib import Path

BASE = 'https://putusan3.mahkamahagung.go.id'
START_URLS = [
    BASE + '/direktori.html',
    BASE + '/direktori/index/kategori/narkotika-dan-psikotropika-1.html',
]
UA = 'AcademicMetadataResearchBot/1.0 (read-only; polite; course assignment)'
MAX_PAGES = 6  # Batas kecil untuk demonstrasi crawler, bukan jaminan 50 hasil
DELAY_SECONDS = 3
TIMEOUT_SECONDS = 20
session = requests.Session()
session.headers.update({'User-Agent': UA, 'Accept': 'text/html'})
logs=[]
robots = RobotFileParser()
robots.set_url(BASE + '/robots.txt')
robots_ready = False
try:
    rr = session.get(BASE + '/robots.txt', timeout=TIMEOUT_SECONDS)
    if rr.status_code == 200 and rr.text.strip():
        robots.parse(rr.text.splitlines())
        robots_ready = True
        print('robots.txt berhasil dibaca; izin akan diperiksa per URL.')
    else:
        print('robots.txt belum dapat dipastikan:', rr.status_code, '- crawling dinonaktifkan.')
except requests.RequestException as exc:
    print('Tidak bisa membaca robots.txt; crawling dinonaktifkan:', type(exc).__name__)
print('Crawling aktif:', robots_ready)

### 10B. Ambil halaman yang diizinkan, batasi jumlah request dan catat statusnya

In [ ]:
def safe_get(url):
    parsed = urlparse(url)
    if parsed.scheme != 'https' or parsed.netloc != urlparse(BASE).netloc:
        logs.append({'url': url, 'status': 'DITOLAK', 'alasan': 'Domain/protokol tidak sesuai'})
        return None
    if not robots_ready or not robots.can_fetch(UA, url):
        logs.append({'url': url, 'status': 'DITOLAK', 'alasan': 'Robots tidak diketahui atau tidak mengizinkan'})
        return None
    try:
        time.sleep(DELAY_SECONDS)
        res = session.get(url, timeout=TIMEOUT_SECONDS, allow_redirects=False)
        if res.status_code != 200:
            logs.append({'url': url, 'status': str(res.status_code), 'alasan': 'HTTP non-200; tidak diteruskan'})
            return None
        if 'html' not in res.headers.get('Content-Type', '').lower():
            logs.append({'url': url, 'status': 'DITOLAK', 'alasan': 'Bukan HTML'})
            return None
        logs.append({'url': url, 'status': '200', 'alasan': 'HTML diambil'})
        return res.text
    except requests.RequestException as exc:
        logs.append({'url': url, 'status': 'ERROR', 'alasan': type(exc).__name__ + ': ' + str(exc)[:120]})
        return None

def local_links(soup, current_url):
    for a in soup.select('a[href]'):
        target=urljoin(current_url, a.get('href',''))
        if urlparse(target).netloc == urlparse(BASE).netloc and urlparse(target).scheme=='https':
            yield a.get_text(' ', strip=True), target

pages={}
queue=list(START_URLS)
seen=set()
while queue and len(seen) < MAX_PAGES:
    url=queue.pop(0)
    if url in seen: continue
    seen.add(url)
    html=safe_get(url)
    if not html:
        continue
    pages[url]=html
    soup=BeautifulSoup(html,'lxml')
    for label, link in local_links(soup,url):
        low=(label+' '+link).lower()
        # Prioritas navigasi kategori dan pengadilan yang sesuai
        if (('kuala-kurun' in low or 'kuala kurun' in low or
             'narkotika-dan-psikotropika' in low) and
             '/direktori/index/' in link and link not in seen and link not in queue):
            queue.append(link)

print('Halaman berhasil dibaca:',len(pages),'dari',len(seen),'percobaan')
print('Contoh status:', logs[:5])

### 10C. Ekstraksi metadata dan tautan putusan dari HTML

Parser ini hanya mengambil informasi yang benar-benar terlihat di halaman. Tidak menebak klasifikasi atau nama pengadilan dari nomor putusan.

In [ ]:
CASE_RE = re.compile(r'(?i)\b(\d{1,5}\s*/\s*Pid\.?\s*Sus\.?\s*/\s*20\d{2}\s*/\s*PN\s*Kkn)\b')

def normalize_case(s):
    return re.sub(r'\s+', '', str(s or '')).replace('.', '').upper()

def parse_metadata(html, source_url):
    soup=BeautifulSoup(html,'lxml')
    output=[]
    # Setiap link putusan pada daftar direktori adalah kandidat, bukan bukti kategori.
    for a in soup.select('a[href]'):
        label=a.get_text(' ',strip=True)
        if 'Putusan' not in label and not CASE_RE.search(label):
            continue
        link=urljoin(source_url,a.get('href',''))
        if '/direktori/putusan/' not in link: continue
        match=CASE_RE.search(label)
        # Jika nomor putusan tidak tertulis, tidak mengklaim kasus PN Kuala Kurun
        if not match: continue
        no=match.group(1)
        if not normalize_case(no).endswith('PNKKN'): continue
        card_text=a.parent.parent.get_text(' ',strip=True)[:1000] if a.parent and a.parent.parent else label
        if 'narkotika' not in card_text.lower():
            klasifikasi='Perlu verifikasi pada halaman detail'
        else:
            klasifikasi='Narkotika dan Psikotropika (berdasarkan teks halaman)'
        output.append({'No Putusan':no,'Lembaga Peradilan':'PN Kuala Kurun',
                       'Klasifikasi di Halaman':klasifikasi,
                       'URL Detail Putusan':link,'URL Sumber Listing':source_url})
    return output

metadata=[]
for url,html in pages.items():
    metadata.extend(parse_metadata(html,url))
crawl_df=pd.DataFrame(metadata,columns=['No Putusan','Lembaga Peradilan','Klasifikasi di Halaman','URL Detail Putusan','URL Sumber Listing'])
if not crawl_df.empty:
    crawl_df['nomor_normalisasi']=crawl_df['No Putusan'].map(normalize_case)
    crawl_df=crawl_df.drop_duplicates('nomor_normalisasi').drop(columns='nomor_normalisasi')
print('Metadata PN Kuala Kurun yang ditemukan:',len(crawl_df))
display(crawl_df.head())

### 10D. Cocokkan metadata website dengan 50 putusan hasil ekstraksi lokal

**Penting:** `Tidak ditemukan di crawling` berarti tidak ditemukan dalam halaman yang sempat diakses, **bukan** berarti tidak ada di MA.

In [ ]:
local_df=full_df[main_cols].copy()
local_df['nomor_kunci']=local_df['No Putusan'].map(normalize_case)
website_df=crawl_df.copy()
website_df['nomor_kunci']=website_df['No Putusan'].map(normalize_case)
website_df=website_df.drop(columns=['No Putusan','Lembaga Peradilan'],errors='ignore')
matching=local_df.merge(website_df,on='nomor_kunci',how='left',validate='one_to_one')
matching['Status Pencocokan']=matching['URL Detail Putusan'].notna().map({True:'Ditemukan pada halaman yang berhasil diakses',False:'Tidak ditemukan pada halaman yang diakses'})
matching=matching.drop(columns=['nomor_kunci'])
log_df=pd.DataFrame(logs,columns=['url','status','alasan'])

crawl_csv=OUT_DIR/'metadata_crawling_PN_Kuala_Kurun.csv'
match_csv=OUT_DIR/'pencocokan_crawling_50_putusan.csv'
log_csv=OUT_DIR/'log_akses_crawling.csv'
crawl_df.to_csv(crawl_csv,index=False,encoding='utf-8-sig')
matching.to_csv(match_csv,index=False,encoding='utf-8-sig')
log_df.to_csv(log_csv,index=False,encoding='utf-8-sig')
report={'timestamp_utc':datetime.now(timezone.utc).isoformat(),
        'robots_terverifikasi':robots_ready,
        'halaman_berhasil':len(pages),
        'metadata_tertemukan':len(crawl_df),
        'data_lokal':len(local_df),
        'data_cocok':int(matching['URL Detail Putusan'].notna().sum()),
        'keterangan':'Metadata dari halaman yang dapat diakses saja; tidak otomatis mewakili seluruh putusan MA.'}
(OUT_DIR/'ringkasan_crawling.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(report,indent=2,ensure_ascii=False))
display(matching[['No Putusan','Status Pencocokan']].head(10))

### 10E. Jalur resmi alternatif — SIPP PN Kuala Kurun dan HTML resmi tersimpan

Hasil percobaan sebelumnya: Direktori MA merespons **403**, sehingga metadata website **0**. Situs resmi PN Kuala Kurun menunjuk ke **SIPP** (Sistem Informasi Penelusuran Perkara). Alamat SIPP PN Kuala Kurun tercatat dalam laporan resmi Pengadilan Tinggi Palangkaraya. SIPP **bukan** Direktori Putusan dan tidak otomatis membuktikan klasifikasi/unggahan putusan pada Dirput; sumber disimpan terpisah.

Notebook berikut hanya membaca **homepage publik** SIPP jika robots.txt dapat diperiksa dan mengizinkan akses. Jika tidak diizinkan/403, crawler berhenti. Sebagai alternatif, kamu boleh membuka halaman publik **secara biasa di browser**, lalu menggunakan `Ctrl+S` → `Webpage, HTML Only` dan mengunggah file HTML tersebut (atau ZIP berisi HTML). File itu ditandai **HTML disimpan manual**, bukan hasil HTTP crawler. Jangan menggunakan login, menghindari CAPTCHA, atau membuat hasil fiktif.

Alamat resmi:
- Website PN: https://pn-kualakurun.go.id/
- SIPP PN: https://sipp.pn-kualakurun.go.id/
- Direktori MA: https://putusan3.mahkamahagung.go.id/direktori.html

Referensi: [Situs PN](https://pn-kualakurun.go.id/), [Laporan PT Palangkaraya 2024](https://www.pt-palangkaraya.go.id/images/dokumenlaporan/LPK_PT_PLK_2024.pdf), [Penjelasan kendala Dirput oleh Kepaniteraan MA](https://kepaniteraan.mahkamahagung.go.id/id/registry-news/2964-sempat-sulit-diakses-kini-dirput-berangsur-membaik).

In [ ]:
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse
from pathlib import Path
from datetime import datetime, timezone
import requests, zipfile, io, re, time

SIPP_HOME = 'https://sipp.pn-kualakurun.go.id/'
SIPP_ROBOTS = 'https://sipp.pn-kualakurun.go.id/robots.txt'
HTML_DIR = WORK / 'html_resmi_tersimpan'
HTML_DIR.mkdir(parents=True, exist_ok=True)
source_html = []   # [{html, sumber, url_asal, metode}]
extra_logs = []

# Hanya satu permintaan halaman setelah aturan robots.txt terverifikasi.
# Status non-200 tidak dianggap sebagai pengambilan metadata.
try:
    rob_res = session.get(SIPP_ROBOTS, timeout=15, allow_redirects=False)
    if rob_res.status_code == 200 and rob_res.text.strip():
        rp = RobotFileParser()
        rp.parse(rob_res.text.splitlines())
        if rp.can_fetch(UA, SIPP_HOME):
            time.sleep(3)
            resp = session.get(SIPP_HOME, timeout=20, allow_redirects=False)
            ok = resp.status_code == 200 and 'html' in resp.headers.get('content-type','').lower()
            extra_logs.append({'url': SIPP_HOME, 'status': str(resp.status_code),
                               'alasan': 'HTML publik terbaca' if ok else 'Tidak dapat mengambil HTML publik'})
            if ok:
                source_html.append({'html': resp.text, 'sumber': 'SIPP PN Kuala Kurun',
                                    'url_asal': SIPP_HOME, 'metode': 'HTTP otomatis diizinkan'})
        else:
            extra_logs.append({'url': SIPP_HOME, 'status': 'DILARANG ROBOTS', 'alasan': 'Tidak melakukan request halaman'})
    else:
        extra_logs.append({'url': SIPP_ROBOTS, 'status': str(rob_res.status_code),
                           'alasan': 'Izin tidak dapat diverifikasi; request halaman dibatalkan'})
except requests.RequestException as exc:
    extra_logs.append({'url': SIPP_ROBOTS, 'status': 'ERROR',
                       'alasan': 'Izin tidak dapat diverifikasi: ' + type(exc).__name__})

print('Jumlah HTML SIPP diambil otomatis:',len(source_html))
print('Log akses sumber alternatif:',extra_logs)

### 10F. Unggah HTML publik yang sudah disimpan dari browser (opsional)

Jika situs MA/SIPP dapat kamu buka **secara normal**, simpan halaman publik hasil daftar putusan atau daftar perkara dengan **Ctrl+S → Webpage, HTML Only** (`.html`). Kamu bisa mengunggah beberapa `.html` atau ZIP berisi file HTML. Pastikan HTML hasil simpan memuat daftar nomor perkara, bukan halaman error/login.

**Jalankan cell di bawah ini hanya jika punya HTML yang disimpan dari browser.** Jika tidak, biarkan `UPLOAD_HTML_MANUAL = False` dan notebook tetap dapat dijalankan. File yang diunggah **bukan** hasil crawling otomatis.

In [ ]:
UPLOAD_HTML_MANUAL = False  # Ubah True jika kamu memiliki file HTML resmi dari browser
if UPLOAD_HTML_MANUAL:
    from google.colab import files
    uploaded_html = files.upload()
    for filename, contents in uploaded_html.items():
        suffix = Path(filename).suffix.lower()
        if suffix in ('.html', '.htm'):
            file_bytes = [(filename,contents)]
        elif suffix == '.zip' and zipfile.is_zipfile(io.BytesIO(contents)):
            with zipfile.ZipFile(io.BytesIO(contents)) as z:
                file_bytes = [(member.filename,z.read(member)) for member in z.infolist()
                              if not member.is_dir() and Path(member.filename).suffix.lower() in ('.html','.htm')]
        else:
            print('Dilewati, bukan HTML/ZIP valid:', filename)
            continue
        for html_name, html_bytes in file_bytes:
            if len(html_bytes)>12_000_000:
                print('Dilewati >12 MB:',html_name); continue
            html_text = html_bytes.decode('utf-8',errors='replace')
            if not any(marker in html_text.lower() for marker in ['putusan','perkara','kuala kurun']):
                print('Periksa file; konten bukan daftar putusan/perkara:',html_name)
                continue
            source_html.append({'html':html_text, 'sumber':'HTML resmi tersimpan',
                               'url_asal':'Tidak direkam otomatis; cocokkan URL sumber secara manual',
                               'metode':'Disimpan manual dari browser: '+Path(html_name).name})
print('Total HTML sumber alternatif:',len(source_html))

### 10G. Ekstraksi metadata secara konservatif dan pencocokan dengan 50 nomor putusan

Ekstraksi otomatis hanya memasukkan **nomor perkara PN Kuala Kurun yang sungguh ditemukan di HTML**, bukan mengasumsikan semua 50 perkara terdaftar. Data dari SIPP dan HTML manual tetap diberi label sumber. **Klasifikasi perkara tidak disimpulkan dari nomor putusan**; klasifikasi narkotika hanya diberi nilai bila kata tersebut ada dalam teks baris sumber.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

CASE_HTML_RE = re.compile(r'(?i)\b(\d{1,5}\s*/\s*Pid\.?\s*Sus\.?\s*/\s*20\d{2}\s*/\s*PN\s*\.?\s*Kkn)\b')

def parse_official_html(html_text, origin, mode, label):
    soup=BeautifulSoup(html_text,'lxml')
    out=[]
    # Baris tabel SIPP, lalu blok hasil daftar Dirput (jika file HTML memang memuatnya)
    nodes = soup.select('tr') + soup.select('article, li, .putusan, .result, .search-result')
    if not nodes:
        nodes = soup.select('body')
    observed=set()
    for node in nodes:
        line=node.get_text(' ',strip=True)
        for match in CASE_HTML_RE.finditer(line):
            case=re.sub(r'\s*/\s*','/',match.group(1)).strip()
            key=normalize_case(case)
            if key in observed:continue
            observed.add(key)
            a=node.find('a',href=True)
            link=urljoin(origin,a['href']) if a and origin.startswith('http') else ''
            category='Narkotika dan Psikotropika (tertulis di sumber)' if re.search(r'(?i)narkotika|psikotropika',line) else 'Tidak dapat dipastikan dari HTML'
            out.append({'No Putusan':case,'Lembaga Peradilan':'PN Kuala Kurun',
                        'Klasifikasi di Halaman':category, 'URL Detail Putusan':link,
                        'URL Sumber Listing': origin if origin.startswith('http') else '',
                        'Sumber Alternatif':label, 'Metode Perolehan':mode,
                        'Bukti Potongan HTML':line[:300]})
    return out

alternative_records=[]
for item in source_html:
    alternative_records.extend(parse_official_html(item['html'],item['url_asal'],item['metode'],item['sumber']))
ALT_COLS=['No Putusan','Lembaga Peradilan','Klasifikasi di Halaman','URL Detail Putusan','URL Sumber Listing',
          'Sumber Alternatif','Metode Perolehan','Bukti Potongan HTML']
alt_df=pd.DataFrame(alternative_records,columns=ALT_COLS)
if not alt_df.empty:
    alt_df['nomor_kunci']=alt_df['No Putusan'].map(normalize_case)
    alt_df=alt_df.drop_duplicates(subset=['nomor_kunci','Sumber Alternatif','Metode Perolehan']).drop(columns=['nomor_kunci'])

local_keys=local_df[['No Putusan']].copy()
local_keys['nomor_kunci']=local_keys['No Putusan'].map(normalize_case)
alt_for_merge=alt_df.copy()
alt_for_merge['nomor_kunci']=alt_for_merge['No Putusan'].map(normalize_case)
alt_keys=set(alt_for_merge['nomor_kunci'])
local_keys['Ditemukan pada HTML alternatif']=local_keys['nomor_kunci'].isin(alt_keys)
local_keys=local_keys.drop(columns=['nomor_kunci'])

alt_csv=OUT_DIR/'metadata_sumber_alternatif.csv'
alt_match_csv=OUT_DIR/'pencocokan_sumber_alternatif_50_putusan.csv'
alt_log_csv=OUT_DIR/'log_sumber_alternatif.csv'
alt_summary_file=OUT_DIR/'ringkasan_sumber_alternatif.json'
alt_df.to_csv(alt_csv,index=False,encoding='utf-8-sig')
local_keys.to_csv(alt_match_csv,index=False,encoding='utf-8-sig')
pd.DataFrame(extra_logs,columns=['url','status','alasan']).to_csv(alt_log_csv,index=False,encoding='utf-8-sig')
alt_summary={'timestamp_utc':datetime.now(timezone.utc).isoformat(),
 'html_sumber_terbaca':len(source_html), 'metadata_alternatif_ditemukan':len(alt_df),
 'dokumen_lokal':len(local_df),'nomor_cocok_dengan_html_alternatif':int(local_keys['Ditemukan pada HTML alternatif'].sum()),
 'penting':'SIPP dan HTML tersimpan tidak setara dengan keberhasilan web crawler Direktori MA; sumber/metode perolehan dibedakan.'}
alt_summary_file.write_text(json.dumps(alt_summary,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(alt_summary,indent=2,ensure_ascii=False))
display(alt_df.head(10))

### 10H. Unduh hasil jalur alternatif

Jika `metadata_alternatif_ditemukan = 0`, **jangan mengarang metadata**. Simpan laporan akses. Untuk keperluan dosen, tuliskan keterbatasan crawler 403 dan tunjukkan dataset PDF/TXT/CSV asli yang benar-benar diproses.

In [ ]:
from google.colab import files
for filepath in [alt_csv, alt_match_csv, alt_log_csv, alt_summary_file]:
    files.download(str(filepath))

### 10I. Unduh laporan crawling Direktori MA

Ketiga CSV tetap dibuat, termasuk ketika akses website ditolak. Dalam kondisi tersebut, CSV metadata akan kosong dan laporan akses menjelaskan penyebabnya; **jangan** mengklaim crawling berhasil.

In [ ]:
from google.colab import files
for fp in [crawl_csv, match_csv, log_csv, OUT_DIR/'ringkasan_crawling.json']:
    files.download(str(fp))

## 11. Checklist final untuk LMS

- [ ] 50 PDF asli pada Google Drive.
- [ ] 50 TXT dan CSV hasil ekstraksi pada Google Drive.
- [ ] Notebook Colab dibagikan sebagai Viewer.
- [ ] Lampirkan `log_akses_crawling.csv` dan `ringkasan_crawling.json` sebagai bukti hasil nyata. Bila situs menolak akses, tulis keterbatasan tersebut, jangan membuat data fiktif.
- [ ] Cocokkan sampel amar/barang bukti dengan PDF asli dan catat hasil pengecekan.
- [ ] Masing-masing anggota kelompok mengumpulkan URL yang diwajibkan dosen.
- [ ] Jika Direktori MA `403`, gunakan catatan log dan alternatif resmi dengan atribusi sumber yang jelas.


---
## 12. Integrasi bukti Direktori MA dari PDF yang disimpan melalui Chrome

**Sumber:** `Daftar_Putusan_Halaman_1.pdf` dan `Daftar_Putusan_Halaman_2.pdf`, dicetak melalui Chrome dari halaman resmi PN Kuala Kurun → Narkotika dan Psikotropika → Putus → 2024.

Tahap ini adalah **parsing arsip halaman yang disimpan manual**, **bukan** bukti bahwa crawler HTTP otomatis berhasil. URL detail diambil dari hyperlink yang memang tertanam di PDF. Data 2024 tidak otomatis mencakup seluruh dataset lokal 50 putusan dari tahun lain.

Bukti pada screenshot halaman detail menunjukkan adanya nomor putusan, tanggal, amar dan barang bukti. Skrip ini **tidak membuat klaim bahwa teks halaman detail sudah di-crawl otomatis**.

In [ ]:
# Jalankan setelah bagian ekstraksi dan crawling sebelumnya, atau sebagai tambahan.
from pathlib import Path
import re, json, pandas as pd, pymupdf
from datetime import datetime, timezone
from urllib.parse import urlparse
from IPython.display import display

WORK = Path('/content/subcpmk3')
OUT_DIR = WORK / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_BUKTI = WORK / 'pdf_bukti_direktori'
PDF_BUKTI.mkdir(parents=True, exist_ok=True)

# Upload dua PDF hasil Ctrl+P Chrome pada halaman daftar tahun 2024.
# Jika sudah ada pada Drive dan tersalin ke PDF_BUKTI, upload tidak perlu diulang.
bukti_files = sorted(PDF_BUKTI.glob('Daftar_Putusan_Halaman_*.pdf'))
if len(bukti_files) < 2:
    from google.colab import files
    uploaded = files.upload()
    for filename, content in uploaded.items():
        if filename.lower().endswith('.pdf'):
            (PDF_BUKTI / Path(filename).name).write_bytes(content)
    bukti_files = sorted(PDF_BUKTI.glob('Daftar_Putusan_Halaman_*.pdf'))
if not bukti_files:
    raise FileNotFoundError('Upload Daftar_Putusan_Halaman_1.pdf dan Daftar_Putusan_Halaman_2.pdf terlebih dahulu.')
print('Bukti PDF tersedia:', [p.name for p in bukti_files])

### 12A. Parsing entri, tanggal dan hyperlink detail dari PDF

Parser tidak mengarang nomor atau URL. Entri yang tidak memiliki tautan akan diberi nilai kosong; sumber PDF/halaman tetap dicatat.

In [ ]:
def norm_case(value):
    m = re.search(r'(\d{1,5})\s*/\s*Pid\.?\s*Sus\.?\s*/\s*(20\d{2})\s*/\s*PN\s*Kkn', str(value), re.I)
    return f"{int(m.group(1))}/Pid.Sus/{m.group(2)}/PN Kkn" if m else ''

CASE_RE_PDF = re.compile(r'Putusan\s+PN\s+Kuala\s+Kurun\s+Nomor\s+(\d{1,5}\s*/\s*Pid\.?Sus\.?\s*/\s*20\d{2}\s*/\s*PN\s*Kkn)',re.I)
DATES_RE = re.compile(r'Register\s*:\s*(\d{2}-\d{2}-\d{4})\s*[—–-]\s*Putus\s*:\s*(\d{2}-\d{2}-\d{4})\s*[—–-]\s*Upload\s*:\s*(\d{2}-\d{2}-\d{4})',re.I)
rows = []
for pdfpath in bukti_files:
    doc = pymupdf.open(str(pdfpath))
    for pi, page in enumerate(doc):
        text = page.get_text('text')
        matches = list(CASE_RE_PDF.finditer(text))
        for j, m in enumerate(matches):
            start = m.start()
            end = matches[j+1].start() if j+1 < len(matches) else min(len(text),start+900)
            segment = text[max(0,start-240):end]
            pre = text[max(0,start-250):start]
            dates = list(DATES_RE.finditer(pre))
            dat = dates[-1].groups() if dates else ('','','')
            defendant = re.search(r'Terdakwa\s*:\s*\n?([^\n]+)', segment,re.I)
            no = norm_case(m.group(1))
            # Link region of corresponding text; only keep official detail URLs.
            rects = page.search_for(m.group(0))
            urls = []
            if rects:
                box = rects[0]
                for link in page.get_links():
                    uri = link.get('uri','')
                    if '/direktori/putusan/' in uri and urlparse(uri).netloc == 'putusan3.mahkamahagung.go.id':
                        if pymupdf.Rect(link['from']).intersects(box):
                            urls.append(uri)
            rows.append({
                'No Putusan':no, 'Tanggal Register':dat[0], 'Tanggal Putus':dat[1],
                'Tanggal Upload':dat[2], 'Terdakwa (daftar)': defendant.group(1).strip() if defendant else '',
                'URL Detail Resmi':urls[0] if urls else '',
                'Metode':'PDF halaman web tersimpan melalui Chrome; parsing lokal',
                'Sumber PDF':pdfpath.name, 'Halaman PDF':pi+1
            })

df_bukti = pd.DataFrame(rows)
if df_bukti.empty:
    raise ValueError('Tidak ada entri terdeteksi. Pastikan PDF merupakan daftar putusan resmi, bukan halaman detail.')
df_bukti = df_bukti.drop_duplicates(subset=['No Putusan']).sort_values('No Putusan').reset_index(drop=True)
print('Entri unik:',len(df_bukti),'| URL detail:',df_bukti['URL Detail Resmi'].ne('').sum())
display(df_bukti.head(8))


### 12B. Pencocokan dengan 50 CSV lokal

Jika `full_df` sudah terbentuk dari sel sebelumnya, gunakan data tersebut. Bila menjalankan bagian ini sendiri, upload `hasil_ekstraksi_putusan_BERSIH.csv`. **Tidak cocok** berarti tidak muncul pada dua halaman daftar tahun 2024 yang diarsipkan, bukan berarti kasus tidak valid.

In [ ]:
if 'full_df' in globals() and 'No Putusan' in full_df.columns:
    df_lokal = full_df.copy()
else:
    csv_candidates = list(OUT_DIR.glob('hasil_ekstraksi_putusan_BERSIH.csv'))
    if not csv_candidates:
        from google.colab import files
        upl = files.upload()
        for fn, content in upl.items():
            if fn.lower().endswith('.csv'):
                (OUT_DIR / Path(fn).name).write_bytes(content)
        csv_candidates = list(OUT_DIR.glob('hasil_ekstraksi_putusan_BERSIH*.csv'))
    if not csv_candidates:
        raise FileNotFoundError('CSV bersih belum tersedia.')
    df_lokal = pd.read_csv(csv_candidates[0])

df_lokal['nomor_kunci'] = df_lokal['No Putusan'].map(norm_case)
df_bukti['nomor_kunci'] = df_bukti['No Putusan'].map(norm_case)
joined = df_lokal.merge(df_bukti.drop(columns=['No Putusan']),on='nomor_kunci',how='left',validate='m:1',indicator=True)
joined['Status Bukti Direktori 2024'] = joined['_merge'].map({'both':'Cocok di PDF daftar 2024','left_only':'Tidak terlihat pada arsip daftar 2024','right_only':'Hanya sumber'}).astype(str)
joined.drop(columns=['_merge'],inplace=True)
print('Data lokal:',len(df_lokal))
print('Data arsip 2024:',len(df_bukti))
print('Data cocok:',joined['Status Bukti Direktori 2024'].eq('Cocok di PDF daftar 2024').sum())
display(joined[['No Putusan','Status Bukti Direktori 2024','Tanggal Putus','URL Detail Resmi']].head(10))

### 12C. Simpan hasil dan ringkasan metode

Catatan untuk laporan dosen: crawling HTTP langsung pernah dicoba namun ditolak 403; halaman daftar tahun 2024 berhasil diakses lewat Chrome dan diarsipkan sebagai PDF; metadata kemudian di-*parse* secara lokal. **Dua metode tidak boleh disamakan.**

In [ ]:
out_meta = OUT_DIR / 'metadata_direktori_2024_dari_PDF_browser.csv'
out_join = OUT_DIR / 'pencocokan_50_putusan_dengan_PDF_browser.csv'
out_report = OUT_DIR / 'ringkasan_arsip_direktori_2024.json'
df_bukti.drop(columns=['nomor_kunci']).to_csv(out_meta,index=False,encoding='utf-8-sig')
joined.to_csv(out_join,index=False,encoding='utf-8-sig')
report = {
  'jumlah_pdf_daftar':len(bukti_files),
  'jumlah_entri_daftar_2024':len(df_bukti),
  'jumlah_url_detail_dari_hyperlink_pdf':int(df_bukti['URL Detail Resmi'].ne('').sum()),
  'jumlah_putusan_lokal':len(df_lokal),
  'jumlah_cocok_dengan_arsip_2024':int(joined['Status Bukti Direktori 2024'].eq('Cocok di PDF daftar 2024').sum()),
  'metode':'Cetak halaman resmi via Chrome sebagai PDF; ekstraksi metadata secara lokal; bukan crawling HTTP otomatis',
  'crawling_otomatis':'Lihat ringkasan_crawling.json; pada pengujian sebelumnya akses situs ditolak HTTP 403',
  'catatan':'Daftar tahun 2024 tidak mewakili putusan lokal dari tahun lain.'
}
out_report.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(report,ensure_ascii=False,indent=2))
from google.colab import files
for p in [out_meta,out_join,out_report]: files.download(str(p))

### 12D. Tahap berikutnya

Simpan kedua PDF bukti dan tiga output baru ke Drive pada folder `Hasil_Ekstraksi` / `Bukti_Direktori_MA`. Jika diperlukan metadata untuk tahun 2025, arsipkan daftar 2025 secara terpisah dan **jangan** menyebut data tahun 2024 sebagai seluruh Direktori MA. Pemeriksaan akurasi isi barang bukti/amar terhadap PDF asli tetap diperlukan.